# Medical Diagnostics — Reusable Scikit-Learn Preprocessing Pipeline

**Goal:** build a leakage-free, reusable `scikit-learn` pipeline for a complex tabular medical-diagnostics dataset, using `ColumnTransformer` to combine numerical scaling with categorical One-Hot and Target Encoding, and doing the train/test split **before** any transformation is fit.

**About the dataset:** this environment has no internet access, so this notebook generates a **synthetic-but-realistic medical diagnostics dataset** (structured like the classic heart-disease risk datasets — mixed numeric vitals/labs, low-cardinality clinical categoricals, and one high-cardinality categorical field) with a fixed random seed for full reproducibility. **The pipeline code is dataset-agnostic** — to use a real dataset (e.g. the UCI/Kaggle Heart Disease dataset, the Pima Indians Diabetes dataset, or a hospital's own extract), simply replace the data-loading cell with `pd.read_csv('your_file.csv')` and update the three column-name lists in Step 2 to match your real columns; nothing else in the pipeline needs to change.

## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

RANDOM_STATE = 42


## Step 0 — Load the dataset

A synthetic medical-diagnostics table: predicting `diagnosis` (1 = disease present) from a mix of numeric vitals/labs, low-cardinality clinical categoricals, and a high-cardinality `hospital_branch` field (15 branches) — deliberately included to motivate **Target Encoding** as an alternative to One-Hot Encoding for high-cardinality columns. Missing values are injected into several columns (as real clinical data almost always has some), to justify imputation inside the pipeline.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 1200

age = rng.normal(54, 9, n).clip(29, 80).round(0)
sex = rng.choice(['male', 'female'], size=n, p=[0.68, 0.32])
chest_pain_type = rng.choice(
    ['typical_angina', 'atypical_angina', 'non_anginal_pain', 'asymptomatic'],
    size=n, p=[0.15, 0.25, 0.25, 0.35]
)
resting_blood_pressure = rng.normal(131, 17, n).clip(94, 200).round(0)
cholesterol = rng.normal(246, 52, n).clip(126, 564).round(0)
fasting_blood_sugar = rng.choice(['yes', 'no'], size=n, p=[0.15, 0.85])
resting_ecg = rng.choice(['normal', 'st_t_abnormality', 'lv_hypertrophy'], size=n, p=[0.5, 0.3, 0.2])
max_heart_rate = rng.normal(150, 23, n).clip(71, 202).round(0)
exercise_induced_angina = rng.choice(['yes', 'no'], size=n, p=[0.33, 0.67])
oldpeak = rng.exponential(1.0, n).clip(0, 6.2).round(1)
num_major_vessels = rng.choice([0, 1, 2, 3], size=n, p=[0.55, 0.2, 0.15, 0.1])
thalassemia = rng.choice(['normal', 'fixed_defect', 'reversible_defect'], size=n, p=[0.55, 0.1, 0.35])
bmi = rng.normal(27.5, 4.5, n).clip(16, 45).round(1)
hospital_branch = rng.choice([f'Branch_{c}' for c in 'ABCDEFGHIJKLMNO'], size=n)

risk_score = (
    0.04 * (age - 54)
    + 0.9 * (chest_pain_type == 'asymptomatic')
    + 0.5 * (thalassemia != 'normal')
    + 0.015 * (cholesterol - 246)
    + 0.02 * (resting_blood_pressure - 131)
    - 0.02 * (max_heart_rate - 150)
    + 0.6 * (exercise_induced_angina == 'yes')
    + 0.35 * num_major_vessels
    + 0.3 * oldpeak
    + 0.3 * (sex == 'male')
    + rng.normal(0, 1, n)
)
prob = 1 / (1 + np.exp(-(risk_score - 2)))
diagnosis = rng.binomial(1, prob)

df = pd.DataFrame({
    'age': age, 'sex': sex, 'chest_pain_type': chest_pain_type,
    'resting_blood_pressure': resting_blood_pressure, 'cholesterol': cholesterol,
    'fasting_blood_sugar': fasting_blood_sugar, 'resting_ecg': resting_ecg,
    'max_heart_rate': max_heart_rate, 'exercise_induced_angina': exercise_induced_angina,
    'oldpeak': oldpeak, 'num_major_vessels': num_major_vessels, 'thalassemia': thalassemia,
    'bmi': bmi, 'hospital_branch': hospital_branch, 'diagnosis': diagnosis,
})

def add_missing(series, frac, rng):
    s = series.copy()
    idx = rng.choice(len(s), size=int(len(s) * frac), replace=False)
    s.iloc[idx] = np.nan
    return s

df['cholesterol'] = add_missing(df['cholesterol'], 0.05, rng)
df['resting_blood_pressure'] = add_missing(df['resting_blood_pressure'], 0.03, rng)
df['bmi'] = add_missing(df['bmi'], 0.04, rng)
df['resting_ecg'] = add_missing(df['resting_ecg'], 0.02, rng)
df['thalassemia'] = add_missing(df['thalassemia'], 0.06, rng)

df.to_csv('medical_diagnostics_synthetic.csv', index=False)
df.head()


    age     sex   chest_pain_type  ...   bmi  hospital_branch diagnosis
0  57.0    male  non_anginal_pain  ...  20.6         Branch_B         0
1  45.0    male      asymptomatic  ...  26.5         Branch_A         1
2  61.0    male  non_anginal_pain  ...  26.0         Branch_D         1
3  62.0    male      asymptomatic  ...  26.9         Branch_N         1
4  36.0  female      asymptomatic  ...  28.6         Branch_J         0

[5 rows x 15 columns]

In [ ]:
print('Shape:', df.shape)
print('\nMissing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nTarget (diagnosis) balance:')
print(df['diagnosis'].value_counts(normalize=True).round(3))


Shape: (1200, 15)

Missing values per column:
resting_blood_pressure    36
cholesterol               60
resting_ecg               24
thalassemia               72
bmi                       48
dtype: int64

Target (diagnosis) balance:
diagnosis
0    0.562
1    0.438
Name: proportion, dtype: float64


## Step 1 — Train/test split **before** any transformation

Fitting a scaler, imputer, or encoder on the full dataset before splitting leaks test-set statistics (means, categories, target-encoding maps) into training — an optimistic bias that won't be there in production. So the split happens **first**, on the raw, untouched DataFrame, and every transformer below is fit only inside the pipeline on `X_train`.

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)
print('\nTrain target balance:')
print(y_train.value_counts(normalize=True).round(3))
print('\nTest target balance:')
print(y_test.value_counts(normalize=True).round(3))


Train shape: (960, 14)  Test shape: (240, 14)

Train target balance:
diagnosis
0    0.562
1    0.438
Name: proportion, dtype: float64

Test target balance:
diagnosis
0    0.562
1    0.438
Name: proportion, dtype: float64


## Step 2 — Identify column groups

- **Numeric** columns → impute + scale.
- **Low-cardinality categorical** columns (2–4 categories each) → impute + One-Hot Encode.
- **High-cardinality categorical** column (`hospital_branch`, 15 categories) → impute + **Target Encode**, which avoids exploding the feature space the way One-Hot would, by replacing each category with a (cross-validated, smoothed) estimate of the mean target for that category.

In [ ]:
numeric_features = ['age', 'resting_blood_pressure', 'cholesterol', 'max_heart_rate',
                     'oldpeak', 'num_major_vessels', 'bmi']
onehot_features = ['sex', 'chest_pain_type', 'fasting_blood_sugar', 'resting_ecg',
                    'exercise_induced_angina', 'thalassemia']
target_encode_features = ['hospital_branch']

print('Numeric features:', numeric_features)
print('One-Hot features:', onehot_features)
print('Target-Encode features:', target_encode_features)
print('\nCardinality of hospital_branch:', X_train['hospital_branch'].nunique())


Numeric features: ['age', 'resting_blood_pressure', 'cholesterol', 'max_heart_rate', 'oldpeak', 'num_major_vessels', 'bmi']
One-Hot features: ['sex', 'chest_pain_type', 'fasting_blood_sugar', 'resting_ecg', 'exercise_induced_angina', 'thalassemia']
Target-Encode features: ['hospital_branch']

Cardinality of hospital_branch: 15


## Step 3 — Per-column-type pipelines

Each `Pipeline` bundles imputation with the appropriate encoding/scaling, so missing values are always handled *before* the encoder or scaler sees the data — and, critically, each step's statistics (medians, categories, target means) will only ever be learned from training data, because the whole thing is fit once, downstream, inside the full pipeline's `.fit(X_train, y_train)`.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

onehot_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

target_encode_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(random_state=RANDOM_STATE)),
])


## Step 4 — Combine into a single `ColumnTransformer`

`ColumnTransformer` routes each named group of columns to its own pipeline and concatenates the results into one feature matrix — this is what lets numeric scaling, one-hot encoding, and target encoding all happen in a single `.fit`/`.transform` call.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat_onehot', onehot_pipeline, onehot_features),
    ('cat_target', target_encode_pipeline, target_encode_features),
])


## Step 5 — Full pipeline: preprocessing + model, fit only on training data

Wrapping the `preprocessor` and the classifier together in one `Pipeline` means `full_pipeline.fit(X_train, y_train)` fits *every* step — imputers, scaler, encoders, and the model — using training data only. Calling `.predict(X_test)` afterwards applies those already-fitted transformations to the test set without ever re-fitting on it, which is exactly how the leakage is avoided.

In [ ]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

full_pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor', ColumnTransformer(...)),
                ('classifier', LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
y_pred = full_pipeline.predict(X_test)
y_proba = full_pipeline.predict_proba(X_test)[:, 1]

print('=== Logistic Regression pipeline — test set performance ===')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))


=== Logistic Regression pipeline — test set performance ===
              precision    recall  f1-score   support

  No Disease       0.63      0.71      0.67       135
     Disease       0.55      0.46      0.50       105

    accuracy                           0.60       240
   macro avg       0.59      0.58      0.58       240
weighted avg       0.59      0.60      0.59       240

ROC-AUC: 0.6657495590828924
Confusion matrix:
 [[96 39]
 [57 48]]


## Step 6 — Cross-validate the *whole pipeline*

Passing the full pipeline (not just the model) to `cross_val_score` keeps the preprocessing leakage-free even under cross-validation: for every fold, scikit-learn refits the imputers, scaler, and encoders (including the target encoder) on that fold's training portion only, before scoring on its held-out portion.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print('5-fold CV ROC-AUC scores:', np.round(cv_scores, 4))
print(f'Mean CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')


5-fold CV ROC-AUC scores: [0.6961 0.7416 0.7519 0.7231 0.6992]
Mean CV ROC-AUC: 0.7224 (+/- 0.0222)


## Step 7 — Reusability: swap only the classifier, keep the same preprocessor

Because the preprocessing lives in its own `ColumnTransformer` object, it can be dropped into a *different* pipeline unchanged — here with a `RandomForestClassifier` instead of logistic regression — without rewriting any imputation/encoding/scaling logic.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE)),
])
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print('=== Random Forest pipeline (same preprocessor) — test set performance ===')
print(classification_report(y_test, rf_pred, target_names=['No Disease', 'Disease']))
print('ROC-AUC:', roc_auc_score(y_test, rf_proba))


=== Random Forest pipeline (same preprocessor) — test set performance ===
              precision    recall  f1-score   support

  No Disease       0.62      0.79      0.69       135
     Disease       0.58      0.38      0.46       105

    accuracy                           0.61       240
   macro avg       0.60      0.58      0.58       240
weighted avg       0.60      0.61      0.59       240

ROC-AUC: 0.6533333333333333


## Step 8 — Persist the trained pipeline for reuse

Saving the *entire* pipeline (preprocessing + model) with `joblib` means a future script can load it and call `.predict()` directly on raw new patient records — no need to re-implement or re-fit any imputation, scaling, or encoding step separately.

In [ ]:
joblib.dump(full_pipeline, 'medical_diagnostics_pipeline.joblib')

feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
print('Saved trained pipeline to medical_diagnostics_pipeline.joblib')
print('\nNumber of features after preprocessing:', len(feature_names))
print(list(feature_names))


Saved trained pipeline to medical_diagnostics_pipeline.joblib

Number of features after preprocessing: 24
['num__age', 'num__resting_blood_pressure', 'num__cholesterol', 'num__max_heart_rate', 'num__oldpeak', 'num__num_major_vessels', 'num__bmi', 'cat_onehot__sex_female', 'cat_onehot__sex_male', 'cat_onehot__chest_pain_type_asymptomatic', 'cat_onehot__chest_pain_type_atypical_angina', 'cat_onehot__chest_pain_type_non_anginal_pain', 'cat_onehot__chest_pain_type_typical_angina', 'cat_onehot__fasting_blood_sugar_no', 'cat_onehot__fasting_blood_sugar_yes', 'cat_onehot__resting_ecg_lv_hypertrophy', 'cat_onehot__resting_ecg_normal', 'cat_onehot__resting_ecg_st_t_abnormality', 'cat_onehot__exercise_induced_angina_no', 'cat_onehot__exercise_induced_angina_yes', 'cat_onehot__thalassemia_fixed_defect', 'cat_onehot__thalassemia_normal', 'cat_onehot__thalassemia_reversible_defect', 'cat_target__hospital_branch']


## Summary

- **Split first:** `train_test_split` was called on the raw `X`/`y` before any imputer, scaler, or encoder ever saw the data.
- **`ColumnTransformer` + `Pipeline`:** numeric columns get median-imputation + standard scaling; low-cardinality categoricals get mode-imputation + One-Hot Encoding; the high-cardinality `hospital_branch` column gets mode-imputation + Target Encoding — all combined into one object.
- **No leakage under CV either:** `cross_val_score` was given the *whole* pipeline, so every fold refits its own imputers/scaler/encoders on that fold's training split only.
- **Reusable:** the same `preprocessor` was dropped into a second pipeline with a different classifier (Random Forest) with zero changes to the preprocessing code, and the final pipeline was persisted with `joblib` for direct reuse on new data.
- **Swapping in real data:** replace the Step 0 cell with `df = pd.read_csv('your_real_dataset.csv')` and update the three column-name lists in Step 2 — Steps 3 onward run unchanged on real datasets such as the UCI/Kaggle Heart Disease dataset or the Pima Indians Diabetes dataset.